# Filter Port Test

A few filters and processes are ported from imagej into python to streamline the workflow. This notebook tests if the ports are successful.

In [ ]:
%load_ext autoreload
%autoreload 2
# %aimport image_processing_tools

## Stripe suppression demo

Demonstrates the stripe suppression filter using Fourier transform. A hard mask (0 and 1) and a soft mask (gradual change) can be applied and compared. The result is as intended - the horizontal stripes are removed, although not perfectly. Only the soft mask approach is applied in the real code.

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import scipy.fft

# def create_synthetic_image(shape=(500, 500)):
#     """Creates an image with a circle and strong horizontal stripes."""
#     rows, cols = shape
#     y, x = np.mgrid[:rows, :cols]
    
#     # 1. Background info (A bright circle)
#     center = (rows // 2, cols // 2)
#     radius = np.sqrt((y - center[0])**2 + (x - center[1])**2)
#     circle = (radius < 100).astype(float) * 0.8
    
#     # 2. Add Horizontal Stripes (Sine wave along Y-axis)
#     stripes = np.sin(y * 0.5) * 0.5
    
#     # Combine
#     image = circle + stripes
#     return image

# def apply_fft_stripe_filter(image, suppress_direction='Horizontal', tolerance=0.05, soft_mask=True):
#     """
#     Applies FFT with Soft or Hard masking to remove stripes.
#     """
#     rows, cols = image.shape
    
#     # 1. Forward FFT (Results in Complex Numbers)
#     fft_original = scipy.fft.fft2(image)
#     fft_shifted = scipy.fft.fftshift(fft_original)
    
#     # 2. Create Coordinate Grid
#     r_idx, c_idx = np.mgrid[0:rows, 0:cols]
#     c_idx = c_idx - cols / 2
#     r_idx = r_idx - rows / 2
    
#     normalized_r = r_idx / rows
#     normalized_c = c_idx / cols
    
#     # 3. Calculate "Wedge" Metrics
#     if suppress_direction == 'Horizontal':
#         dist_from_axis = np.abs(normalized_c)
#         dist_along_axis = np.abs(normalized_r)
#     else:
#         dist_from_axis = np.abs(normalized_r)
#         dist_along_axis = np.abs(normalized_c)

#     dist_along_axis[dist_along_axis == 0] = 1e-10
#     ratio = dist_from_axis / dist_along_axis
    
#     # 4. Generate Mask
#     if soft_mask:
#         # --- SOFT MASK (Gaussian) ---
#         sigma = tolerance
#         mask = 1.0 - np.exp(-(ratio**2) / (2 * sigma**2))
#     else:
#         # --- HARD MASK (Binary) ---
#         # FIX: Explicitly use dtype=float so the mask is not complex128
#         mask = np.ones(fft_shifted.shape, dtype=float)
#         mask[ratio < tolerance] = 0

#     # 5. Protect Center (DC Component)
#     center_r, center_c = rows // 2, cols // 2
#     mask[center_r-2:center_r+3, center_c-2:center_c+3] = 1.0
    
#     # 6. Apply Filter & Inverse FFT
#     # Note: mask is float, fft_shifted is complex. Result is complex.
#     fft_filtered = fft_shifted * mask
    
#     ifft_shifted = scipy.fft.ifftshift(fft_filtered)
#     image_filtered = scipy.fft.ifft2(ifft_shifted).real
    
#     return fft_shifted, mask, fft_filtered, image_filtered

# # --- Execution ---

# # 1. Generate Image
# img = create_synthetic_image()

# # 2. Process with SOFT Mask
# fft_orig, mask_viz, fft_filt, img_filt = apply_fft_stripe_filter(
#     img, 
#     suppress_direction='Horizontal', 
#     tolerance=0.05, 
#     soft_mask=True  # <--- Set to False to see the difference
# )

# # --- Visualization ---

# fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# # Helper to plot spectra
# def plot_spectrum(ax, data, title, is_mask=False):
#     if is_mask:
#         # Plot mask linearly (0 to 1)
#         im = ax.imshow(data, cmap='gray', vmin=0, vmax=1)
#     else:
#         # Plot spectrum logarithmically
#         spectrum = np.log(1 + np.abs(data))
#         im = ax.imshow(spectrum, cmap='inferno')
#     ax.set_title(title)
#     ax.axis('off')
#     return im

# # Row 1: Original
# axes[0, 0].imshow(img, cmap='gray')
# axes[0, 0].set_title("Original Image")
# axes[0, 0].axis('off')

# plot_spectrum(axes[0, 1], fft_orig, "Original FFT (Note Vertical Line)")

# # Plot the Mask itself to see the "Softness"
# im_mask = plot_spectrum(axes[0, 2], mask_viz, "Soft Filter Mask", is_mask=True)
# plt.colorbar(im_mask, ax=axes[0, 2], fraction=0.046, pad=0.04)

# # Row 2: Filtered
# axes[1, 0].imshow(img_filt, cmap='gray')
# axes[1, 0].set_title("Filtered Image (Soft Mask)")
# axes[1, 0].axis('off')

# plot_spectrum(axes[1, 1], fft_filt, "Filtered FFT (Blurred Vertical Line)")

# # Detail view of mask (Zoom in center)
# h, w = mask_viz.shape
# zoom_size = 50
# zoom_slice = np.s_[h//2 - zoom_size : h//2 + zoom_size, w//2 - zoom_size : w//2 + zoom_size]
# axes[1, 2].imshow(mask_viz[zoom_slice], cmap='gray', vmin=0, vmax=1)
# axes[1, 2].set_title("Zoom on Mask Center (See Gradient)")
# axes[1, 2].axis('off')

# plt.tight_layout()
# plt.show()

## Testing filters

In [ ]:
from image_processing_tools.image_class.image_filters import ImageJProcessor
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI",
    "~/A8/Data_Roan/251205_MonoCa9_Phalloidin_Cellbrite_LowConf/Ca922_Mono_LowConf_Phal_Cellbrite_02_CY5, FITC, DAPI, BF",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_DIC",
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SHARPEST-manual*.tif",
    "SUBTRACT-direct*O5*.tif"
    )

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[1], file_pattern[2], verbose=True)

# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/filter_port_test")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

print(f"The current output dir is {current_output_dir}")

In [ ]:
img_c = ImageContainer(files[0],config)
img = img_c.merge()
pipeline = ImageJProcessor(img)
pipeline.reset()

# ij_clahe = ImageContainer(files[1],config)
# ij_clahe_img = ij_clahe.merge()

# ij_fft = ImageContainer(files[3],config)
# ij_fft_img = ij_fft.merge()

# ij_clahe_fast = ImageContainer(files[7],config)
# ij_clahe_fast_img = ij_clahe_fast.merge()

In [ ]:
pipeline.reset()
img_clahe = pipeline.enhance_contrast_clahe(block_size=127, slope=2.5, bins=256)
img_clahe_ga2 = pipeline.gaussian_blur(sigma = 2)
img_clahe_ga2_fftbpf = pipeline.fft_bandpass_filter(large_structures_down_to=15, small_structures_up_to=3, suppress_stripes='None', tolerance=0.05)
img_clahe_ga2_fftbpf_rollingball = pipeline.imagej_rolling_ball(radius=50, light_background=False, use_paraboloid=False)
vesselness, scale_map = pipeline.frangi_imagej_ops(scales=[2,4],c=15,beta=0.7)

### Compare CLAHE

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import gc

# output_filename = current_output_dir / f"{Path(img_c.name).stem}_clahe_compare.png"

# fig, axes = plt.subplots(1,4,figsize=(16,4))

# axes[0].imshow(img,cmap='gray')
# axes[0].set_title('Original')
# axes[0].axis('off')

# axes[1].imshow(ij_clahe_img, cmap='gray')
# axes[1].set_title('ImageJ CLAHE')
# axes[1].axis('off')

# axes[2].imshow(ij_clahe_fast_img, cmap='gray')
# axes[2].set_title('Fast ImageJ CLAHE')
# axes[2].axis('off')

# axes[3].imshow(img_clahe,cmap='gray')
# axes[3].set_title('Python CLAHE')
# axes[3].axis('off')

# plt.tight_layout()
# plt.show()

# plt.savefig(output_filename, dpi=300, bbox_inches='tight')
# plt.close()
# gc.collect()
# print(f"Saved: {output_filename}")

### Compare FFT

In [ ]:
# import matplotlib.pyplot as plt
# import gc

# output_filename = current_output_dir / f"{Path(img_c.name).stem}_fftbpf_compare.png"

# fig, axes = plt.subplots(1,3,figsize=(12,4))

# axes[0].imshow(img,cmap='gray')
# axes[0].set_title('Original')
# axes[0].axis('off')

# axes[1].imshow(ij_fft_img, cmap='gray')
# axes[1].set_title('ImageJ FFT BPF')
# axes[1].axis('off')

# axes[2].imshow(img_clahe_ga2_fftbpf,cmap='gray')
# axes[2].set_title('Python FFT BPF')
# axes[2].axis('off')

# plt.tight_layout()
# plt.show()

# plt.savefig(output_filename, dpi=300, bbox_inches='tight')
# plt.close()
# gc.collect()
# print(f"Saved: {output_filename}")

### Debug rollingball

In [ ]:
# from image_processing_tools.image_class.rollingball_debug import BackgroundSubtracter
# import gc
# import matplotlib.pyplot as plt

# output_filename = current_output_dir / f"{Path(img_c.name).stem}_rolling_debug.png"

# bs = BackgroundSubtracter()

# result = bs.rolling_ball_background(
#     img_clahe_ga2_fftbpf, 
#     radius=30, 
#     light_background=False, 
#     use_paraboloid=False,
#     debug=True  # <--- Shows the requested plots
# )

# plt.savefig(output_filename, dpi=300, bbox_inches='tight')
# plt.close()
# gc.collect()
# print(f"Saved: {output_filename}")

### Overview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gc
from image_processing_tools.util.phalloidin_processing import invert_image_colors, get_otsu_edge_map

output_filename = current_output_dir / f"{Path(img_c.name).stem}_filter_test.png"

# Create a figure with subplots (2 rows, 4 columns)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# List of images to plot with their titles and colormaps
# Assuming 'img' is the variable name for the original image
plot_data = [
    (img, "Original Image", 'gray'),
    (img_clahe, "CLAHE", 'gray'),
    (img_clahe_ga2, "Gaussian Blur", 'gray'),
    (img_clahe_ga2_fftbpf, "FFT Bandpass", 'gray'),
    (img_clahe_ga2_fftbpf_rollingball, "Rolling Ball BG Sub", 'gray'),
    (invert_image_colors(get_otsu_edge_map(vesselness)), "Frangi Vesselness", 'gray'),
    (scale_map, "Frangi Scale Map", 'jet')
]

# Iterate and plot
for i, (image, title, cmap) in enumerate(plot_data):
    ax = axes[i]
    
    # --- Contrast Enhancement Fix ---
    # Calculate robust limits (1st and 99th percentiles) to ignore outliers.
    # We only apply this to 'gray' images, leaving the 'jet' scale map (heatmap) absolute.
    vmin, vmax = None, None
    # if cmap == 'gray':
    #     # Check if image is not empty or constant to avoid errors
    #     if image.min() != image.max():
    #         vmin, vmax = np.percentile(image, (0.5, 99.5))
    
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')
    
    # Add a colorbar specifically for the Scale Map
    if title == "Frangi Scale Map":
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Scale (px)')

# Hide the last empty subplot (since we have 7 images and 8 slots)
axes[-1].axis('off')

plt.tight_layout()
plt.show()

plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")

In [ ]:
pipeline.reset()

## Another sequence

In [ ]:
pipeline.reset()
img_fftbpf = pipeline.fft_bandpass_filter(large_structures_down_to=60, small_structures_up_to=3, suppress_stripes='Both', tolerance=0.05)
img_fftbpf_ga2 = pipeline.gaussian_blur(sigma = 2)
img_fftbpf_ga2_rollingball = pipeline.imagej_rolling_ball(radius=50, light_background=False, use_paraboloid=False)
img_fftbpf_ga2_rollingball_clahe = pipeline.enhance_contrast_clahe(block_size=127, slope=3, bins=256)
vesselness, scale_map = pipeline.frangi_imagej_ops(scales=[8,10], beta=0.1, c=10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gc

output_filename = current_output_dir / f"{Path(img_c.name).stem}_filter_test_sequence2.png"

# Create a figure with subplots (2 rows, 4 columns)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# List of images to plot with their titles and colormaps
# Assuming 'img' is the variable name for the original image
plot_data = [
    (img, "Original Image", 'gray'),
    (img_fftbpf, "FFT Bandpass", 'gray'),
    (img_fftbpf_ga2, "Gaussian Blur", 'gray'),
    (img_fftbpf_ga2_rollingball, "Rolling Ball BG Sub", 'gray'),
    (img_fftbpf_ga2_rollingball_clahe, "CLAHE", 'gray'),
    (vesselness, "Frangi Vesselness", 'gray'),
    (scale_map, "Frangi Scale Map", 'jet')
]

# Iterate and plot
for i, (image, title, cmap) in enumerate(plot_data):
    ax = axes[i]
    
    # --- Contrast Enhancement Fix ---
    # Calculate robust limits (1st and 99th percentiles) to ignore outliers.
    # We only apply this to 'gray' images, leaving the 'jet' scale map (heatmap) absolute.
    vmin, vmax = None, None
    if cmap == 'gray':
        # Check if image is not empty or constant to avoid errors
        if image.min() != image.max():
            vmin, vmax = np.percentile(image, (0.5, 99.5))
    
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')
    
    # Add a colorbar specifically for the Scale Map
    if title == "Frangi Scale Map":
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Scale (px)')

# Hide the last empty subplot (since we have 7 images and 8 slots)
axes[-1].axis('off')

plt.tight_layout()
plt.show()

plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.close()
gc.collect()
print(f"Saved: {output_filename}")